In [1]:
import torch
import gpytorch
import pandas as pd
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.utils.data import TensorDataset, DataLoader
from pyproj import Transformer
from sklearn.metrics import pairwise_distances
from scipy.interpolate import RegularGridInterpolator
from torch_geometric.data import Data




/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#read in dataset. 
with open('imputed_study_data.pkl', 'rb') as f:
    df=pickle.load(f)

In [3]:
#convert to daily values 
df['date'] = df['time'].dt.tz_convert(None).dt.date
agg_cols = [
    'value',
    'temperature_2m','relative_humidity_2m','dew_point_2m',
    'wind_speed_10m','wind_direction_10m','surface_pressure','precipitation'
]
grouped = df.groupby(['location_id','station_lat','station_lon','date'], as_index=False)[agg_cols].mean()

grouped.head(5)

,location_id,station_lat,station_lon,date,value,temperature_2m,relative_humidity_2m,dew_point_2m,wind_speed_10m,wind_direction_10m,surface_pressure,precipitation
0,2622586,37.580167,127.044856,2024-12-01,20.541667,0.770833,95.375000,0.062500,2.983333,94.583333,999.987500,0.012500
1,2622586,37.580167,127.044856,2024-12-02,21.416667,4.108333,83.958333,1.487500,7.937500,206.416667,999.816667,0.020833
2,2622586,37.580167,127.044856,2024-12-03,6.583333,-0.920833,59.791667,-8.079167,6.525000,269.875000,1007.179167,0.000000
3,2622586,37.580167,127.044856,2024-12-04,11.128392,0.145833,66.041667,-5.887500,4.466667,290.041667,1006.158333,0.000000
4,2622586,37.580167,127.044856,2024-12-05,8.666667,1.191667,72.083333,-3.695833,5.216667,256.791667,1001.008333,0.000000


In [12]:
#now build the network edge list. 
#we build one network per day. 
#edges are treated as continuous values between zero and one.
#first the RBF kernel is used to compute similarity and then KNN is applied to sparsify the graph. 


# --- CONFIG ---
tau = 0.1   # temperature for softmax (controls sharpness)
sigma = 1.0

predictor_cols = [
    'temperature_2m', 'relative_humidity_2m', 'dew_point_2m',
    'wind_speed_10m', 'wind_direction_10m', 'surface_pressure',
    'precipitation'
]

def rbf_similarity(X, sigma=1.0):
    D = pairwise_distances(X, metric='euclidean')
    return np.exp(-(D**2) / (2 * sigma**2))

def soft_knn_adjacency(S, tau=0.1):
    expS = np.exp(S / tau)
    P = expS / expS.sum(axis=1, keepdims=True)
    return 0.5 * (P + P.T)   # symmetric

# --- BUILD DAILY GRAPHS ---
daily_graphs = {}

for date, df_day in grouped.groupby('date'):
    date_str = pd.to_datetime(date).strftime('%Y-%m-%d')   # <-- FIX HERE

    node_ids = df_day['location_id'].values
    X = df_day[predictor_cols].values

    S = rbf_similarity(X, sigma=sigma)
    A = soft_knn_adjacency(S, tau=tau)

    daily_graphs[date_str] = {
        'node_ids': node_ids,
        'X': X,
        'A': A
    }



In [17]:
#now make continuous fields. 



predictor_cols = [
    'temperature_2m', 'relative_humidity_2m', 'dew_point_2m',
    'wind_speed_10m', 'wind_direction_10m', 'surface_pressure',
    'precipitation'
]

def build_continuous_fields(grouped, grid_res=100):
    """
    Build continuous predictor fields for each day using bilinear interpolation.
    Returns: dict[date_str] -> {
        'interpolators': {predictor: callable(lat, lon)},
        'sensor_data': original df for that day
    }
    """
    continuous_fields = {}

    # Extract global bounding box
    all_lats = grouped['station_lat'].values
    all_lons = grouped['station_lon'].values

    lat_min, lat_max = all_lats.min(), all_lats.max()
    lon_min, lon_max = all_lons.min(), all_lons.max()

    # Build regular grid
    lat_grid = np.linspace(lat_min, lat_max, grid_res)
    lon_grid = np.linspace(lon_min, lon_max, grid_res)

    for date, df_day in grouped.groupby('date'):
        date_str = pd.to_datetime(date).strftime('%Y-%m-%d')

        # Extract sensor coordinates + predictors
        lats = df_day['station_lat'].values
        lons = df_day['station_lon'].values

        interpolators = {}

        # Build interpolator for each predictor
        for col in predictor_cols:
            values = df_day[col].values

            # Interpolate onto regular grid
            # We use scipy's griddata for scattered -> grid
            from scipy.interpolate import griddata
            grid_vals = griddata(
                points=np.column_stack([lats, lons]),
                values=values,
                xi=np.meshgrid(lat_grid, lon_grid, indexing='ij'),
                method='linear'
            )

            # Fill NaNs (outside convex hull) with nearest
            nan_mask = np.isnan(grid_vals)
            if nan_mask.any():
                grid_vals[nan_mask] = griddata(
                    points=np.column_stack([lats, lons]),
                    values=values,
                    xi=np.meshgrid(lat_grid, lon_grid, indexing='ij'),
                    method='nearest'
                )[nan_mask]

            # Build bilinear interpolator
            interp = RegularGridInterpolator(
                (lat_grid, lon_grid),
                grid_vals,
                bounds_error=False,
                fill_value=None
            )

            interpolators[col] = interp

        # Store everything
        continuous_fields[date_str] = {
            'interpolators': interpolators,
            'sensor_data': df_day
        }

    return continuous_fields


# --- BUILD THE CONTINUOUS FIELDS ---
continuous_fields = build_continuous_fields(grouped)



In [20]:
#now train a graph convolutional model on the daily graphs. 



# daily_graphs: {date_str: {'node_ids', 'X', 'A'}}
# grouped: your original dataframe with 'date', 'location_id', 'value'

def build_spatiotemporal_samples(daily_graphs, grouped, start_date, end_date):
    """
    Build (X_t, edge_index, edge_weight, y_{t+1}) samples between start_date and end_date (inclusive).
    Dates are 'YYYY-MM-DD' strings.
    """
    # Filter and sort dates
    all_dates = sorted(daily_graphs.keys())
    all_dates = [d for d in all_dates if start_date <= d <= end_date]

    # Build mapping date -> PM2.5 per location_id
    pm25_by_date = {}
    for date, df_day in grouped.groupby('date'):
        date_str = pd.to_datetime(date).strftime('%Y-%m-%d')
        if date_str in all_dates:
            pm25_by_date[date_str] = df_day.set_index('location_id')['value']

    samples = []

    for i in range(len(all_dates) - 1):
        d_t = all_dates[i]
        d_tp1 = all_dates[i + 1]

        g_t = daily_graphs[d_t]
        node_ids_t = g_t['node_ids']
        X_t = g_t['X']
        A_t = g_t['A']

        # PM2.5 at t and t+1 aligned by location_id
        pm25_t = pm25_by_date[d_t].reindex(node_ids_t).values
        pm25_tp1 = pm25_by_date[d_tp1].reindex(node_ids_t).values  # assume same nodes; reindex handles order

        # Input node features: predictors_t + pm25_t
        X_in = np.concatenate([X_t, pm25_t[:, None]], axis=1)  # shape (N, D+1)

        # Build edge_index and edge_weight from A_t
        src, dst = np.nonzero(A_t)
        edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
        edge_weight = torch.tensor(A_t[src, dst], dtype=torch.float32)

        # Targets: PM2.5 at t+1
        y = torch.tensor(pm25_tp1, dtype=torch.float32)

        data = Data(
            x=torch.tensor(X_in, dtype=torch.float32),
            edge_index=edge_index,
            edge_weight=edge_weight,
            y=y,
        )
        data.node_ids = node_ids_t
        data.date_t = d_t
        data.date_tp1 = d_tp1

        samples.append(data)

    return samples

# Example: train on December 2024
train_samples = build_spatiotemporal_samples(
    daily_graphs,
    grouped,
    start_date='2024-12-01',
    end_date='2024-12-31'
)


In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class SpatioTemporalGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels=1):
        super().__init__()
        self.gcn = GCNConv(in_channels, hidden_channels)
        self.gru = nn.GRU(hidden_channels, hidden_channels, batch_first=True)
        self.lin = nn.Linear(hidden_channels, out_channels)

    def forward(self, data_seq):
        """
        data_seq: list of Data objects [day_t, day_t_minus1, ...] or just [day_t]
        For now, we assume length-1 sequences: [data_t].
        """
        # Stack node features for sequence dimension
        # Here: seq_len = 1, so we just add a dimension
        x = data_seq[0].x
        edge_index = data_seq[0].edge_index
        edge_weight = data_seq[0].edge_weight

        # Spatial GNN
        h = F.relu(self.gcn(x, edge_index, edge_weight))

        # Add sequence dimension: (batch=1, seq_len=1, nodes*feat) is awkward,
        # so we treat nodes as "batch" for GRU: (nodes, seq_len, feat)
        h_seq = h.unsqueeze(1)  # (N, 1, hidden)

        # GRU over time dimension (here length 1, but extensible)
        h_out, _ = self.gru(h_seq)  # (N, 1, hidden)
        h_last = h_out[:, -1, :]    # (N, hidden)

        # Node-wise prediction
        y_hat = self.lin(h_last).squeeze(-1)  # (N,)
        return y_hat


In [22]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

in_channels = train_samples[0].x.shape[1]   # predictors + pm25_t
hidden_channels = 64

model = SpatioTemporalGNN(in_channels, hidden_channels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

def train_epoch(samples):
    model.train()
    total_loss = 0.0

    for data in samples:
        data = data.to(device)
        optimizer.zero_grad()

        # For now, sequence length = 1: [data_t]
        y_hat = model([data])
        loss = loss_fn(y_hat, data.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(samples)

# Example: train for a few epochs
for epoch in range(1, 51):
    loss = train_epoch(train_samples)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Train MSE: {loss:.4f}")


Epoch 010 | Train MSE: 219.5820
Epoch 020 | Train MSE: 127.2233
Epoch 030 | Train MSE: 94.6341
Epoch 040 | Train MSE: 85.2630
Epoch 050 | Train MSE: 83.0491


In [23]:
train_samples_jan = build_spatiotemporal_samples(
    daily_graphs,
    grouped,
    start_date='2025-01-01',
    end_date='2025-01-31'
)


In [11]:
# requirements:
# pip install requests pandas numpy tqdm pyarrow pyproj

import os
import time
import math
import random
import requests
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import timedelta, datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyproj import Transformer
from threading import Lock

# -------------------------
# User inputs (edit as needed)
# -------------------------
BBOX = (128.8, 34.8, 129.3, 35.3)   # (min_lon, min_lat, max_lon, max_lat)
lengthscale_km = 4.5                 # spatial lengthscale in km (used to set spacing if building grid)
spacing_km = lengthscale_km / 2.0
cell_size_m = spacing_km * 1000.0

# Try to use existing grid_centers or centers; otherwise build grid from BBOX
try:
    grid_centers  # noqa: F821
    use_existing_grid = True
except NameError:
    try:
        centers  # noqa: F821
        grid_centers = centers
        use_existing_grid = True
    except NameError:
        use_existing_grid = False

# -------------------------
# Open‑Meteo daily endpoint and variables (deduped)
# -------------------------
ARCHIVE_BASE = "https://archive-api.open-meteo.com/v1/archive"
daily_vars = [
    "temperature_2m_mean",
    "relative_humidity_2m_mean",
    "dew_point_2m_mean",
    "windspeed_10m_mean",
    "winddirection_10m_dominant",
    "surface_pressure_mean",
    "precipitation_sum"
]
# ensure uniqueness while preserving order
_seen = set()
daily_vars = [x for x in daily_vars if not (x in _seen or _seen.add(x))]

timezone = "UTC"

# Date range (uses your provided last timestamp)
first_timestamp = pd.Timestamp('2024-12-01 00:00:00+0000', tz='UTC')
start_date = first_timestamp.tz_convert("UTC").date().isoformat()
last_timestamp = pd.Timestamp("2025-08-31 23:00:00+0000", tz="UTC")
end_date = last_timestamp.tz_convert("UTC").date().isoformat()

# Output and fetch settings (safer defaults)
out_dir = "open_meteo_grid_daily_direct"
os.makedirs(out_dir, exist_ok=True)
requests_per_second = 1.0   # safer default; increase only if you confirm API capacity
max_workers = 4            # lower concurrency to reduce bursts
max_retries = 5
retry_backoff_base = 2.0   # base for exponential backoff
max_backoff_seconds = 300  # cap backoff to 5 minutes

# -------------------------
# Build grid_centers if needed (square metric grid using AEQD projection)
# -------------------------
if not use_existing_grid:
    min_lon, min_lat, max_lon, max_lat = BBOX
    center_lon = (min_lon + max_lon) / 2.0
    center_lat = (min_lat + max_lat) / 2.0

    proj_str = f"+proj=aeqd +lat_0={center_lat} +lon_0={center_lon} +units=m +datum=WGS84 +no_defs"
    transformer_to_m = Transformer.from_crs("epsg:4326", proj_str, always_xy=True)
    transformer_to_lonlat = Transformer.from_crs(proj_str, "epsg:4326", always_xy=True)

    def lonlat_to_m(lon, lat):
        x, y = transformer_to_m.transform(lon, lat)
        return float(x), float(y)

    def m_to_lonlat(x, y):
        lon, lat = transformer_to_lonlat.transform(x, y)
        return float(lon), float(lat)

    x_min, y_min = lonlat_to_m(min_lon, min_lat)
    x_max, y_max = lonlat_to_m(max_lon, max_lat)
    x0, x1 = min(x_min, x_max), max(x_min, x_max)
    y0, y1 = min(y_min, y_max), max(y_min, y_max)

    n_cols = int(math.ceil((x1 - x0) / cell_size_m))
    n_rows = int(math.ceil((y1 - y0) / cell_size_m))
    grid_centers = []
    for i in range(n_cols):
        for j in range(n_rows):
            x_left = x0 + i * cell_size_m
            x_right = x0 + (i + 1) * cell_size_m
            y_bottom = y0 + j * cell_size_m
            y_top = y0 + (j + 1) * cell_size_m
            cx = (x_left + x_right) / 2.0
            cy = (y_bottom + y_top) / 2.0
            lonc, latc = m_to_lonlat(cx, cy)
            grid_centers.append((lonc, latc))
    print(f"Built grid: {n_cols} cols × {n_rows} rows = {len(grid_centers)} cells; spacing {spacing_km:.3f} km")
else:
    # normalize grid_centers formats
    if isinstance(grid_centers, np.ndarray):
        grid_centers = [tuple(x) for x in grid_centers.tolist()]
    elif isinstance(grid_centers, pd.DataFrame):
        if {'lon','lat'}.issubset(set(grid_centers.columns)):
            grid_centers = list(zip(grid_centers['lon'].values, grid_centers['lat'].values))
        else:
            raise RuntimeError("If grid_centers is a DataFrame it must contain 'lon' and 'lat' columns.")
    print(f"Using existing grid_centers with {len(grid_centers)} cells")

# Quick validation: ensure tuples are (lon, lat). If many entries look swapped, offer automatic swap.
def looks_like_latlon_swapped(sample):
    lon, lat = sample
    return abs(lon) <= 90 and abs(lat) <= 180 and not (-180 <= lon <= 180 and -90 <= lat <= 90)

if len(grid_centers) > 0:
    first = grid_centers[0]
    if looks_like_latlon_swapped(first):
        print("Detected grid_centers likely in (lat, lon) order; swapping to (lon, lat).")
        grid_centers = [(lonlat[1], lonlat[0]) for lonlat in grid_centers]

# -------------------------
# Helpers for daily fetch with retries, 429 handling, and caching
# -------------------------
def build_params_daily(lat, lon, start_iso, end_iso, daily_vars, timezone="UTC"):
    return {
        "latitude": float(lat),
        "longitude": float(lon),
        "start_date": start_iso,
        "end_date": end_iso,
        "daily": ",".join(daily_vars),
        "timezone": timezone
    }

SESSION = requests.Session()
RATE_LOCK = Lock()
LAST_REQUEST_TS = 0.0

def throttle():
    """Global pacing to respect requests_per_second across threads."""
    global LAST_REQUEST_TS
    with RATE_LOCK:
        min_interval = 1.0 / max(1.0, requests_per_second)
        now = time.time()
        wait = LAST_REQUEST_TS + min_interval - now
        if wait > 0:
            time.sleep(wait)
        LAST_REQUEST_TS = time.time()

def _sleep_with_jitter(seconds):
    """Sleep with small jitter to avoid synchronized retries."""
    jitter = random.uniform(0.0, 0.25 * seconds) if seconds > 0 else 0.0
    time.sleep(seconds + jitter)

def fetch_daily_with_retries(lat, lon, start_iso, end_iso, daily_vars, timezone="UTC"):
    params = build_params_daily(lat, lon, start_iso, end_iso, daily_vars, timezone)
    attempt = 0
    while attempt <= max_retries:
        try:
            throttle()
            r = SESSION.get(ARCHIVE_BASE, params=params, timeout=60)
            if r.status_code == 200:
                try:
                    return r.json()
                except ValueError:
                    raise RuntimeError(f"Invalid JSON response for ({lat},{lon})")
            elif r.status_code == 429:
                # Rate limited: honor Retry-After if present, otherwise exponential backoff
                retry_after = r.headers.get("Retry-After")
                if retry_after is not None:
                    try:
                        wait = float(retry_after)
                    except Exception:
                        # sometimes Retry-After is a HTTP-date; fallback to a safe wait
                        wait = min(60.0, retry_backoff_base ** (attempt + 1))
                    wait = min(wait, max_backoff_seconds)
                    print(f"429 for ({lat},{lon}) — server asked to wait {wait}s (Retry-After). Attempt {attempt+1}/{max_retries}")
                    _sleep_with_jitter(wait)
                else:
                    # exponential backoff with jitter
                    wait = min(max_backoff_seconds, retry_backoff_base ** (attempt + 1))
                    print(f"429 for ({lat},{lon}) — backing off {wait}s (no Retry-After). Attempt {attempt+1}/{max_retries}")
                    _sleep_with_jitter(wait)
                attempt += 1
                continue
            else:
                text = r.text[:1000] if r.text else ""
                print(f"API returned status {r.status_code} for ({lat},{lon}) start={start_iso} end={end_iso}: {text}")
                r.raise_for_status()
        except requests.RequestException as e:
            attempt += 1
            # exponential backoff with jitter for network errors
            wait = min(max_backoff_seconds, retry_backoff_base ** attempt)
            print(f"Network/request error for ({lat},{lon}) attempt {attempt}/{max_retries}: {e}. Backing off {wait}s")
            _sleep_with_jitter(wait)
            if attempt > max_retries:
                raise RuntimeError(f"Failed to fetch daily ({lat},{lon}) after {max_retries} retries: {e}")
        except Exception as e:
            attempt += 1
            wait = min(max_backoff_seconds, retry_backoff_base ** attempt)
            print(f"Unexpected error for ({lat},{lon}) attempt {attempt}/{max_retries}: {e}. Backing off {wait}s")
            _sleep_with_jitter(wait)
            if attempt > max_retries:
                raise RuntimeError(f"Failed to fetch daily ({lat},{lon}) after {max_retries} retries: {e}")
    raise RuntimeError("unreachable")

def safe_daily_path(cell_dir, idx):
    return os.path.join(cell_dir, f"cell_{idx:04d}_daily")

# -------------------------
# Parse daily payload into DataFrame (handles missing vars gracefully)
# -------------------------
def parse_daily_payload(payload, daily_vars):
    daily = payload.get("daily", {})
    times = daily.get("time", [])
    if len(times) == 0:
        return pd.DataFrame(columns=["date"] + daily_vars)
    try:
        df = pd.DataFrame({"date": pd.to_datetime(times)})
        for var in daily_vars:
            vals = daily.get(var)
            if vals is None:
                df[var] = np.nan
            else:
                df[var] = pd.to_numeric(vals, errors="coerce")
        df['date'] = pd.to_datetime(df['date']).dt.date
        return df
    except Exception as e:
        print("Failed to parse payload into DataFrame:", e)
        try:
            print("Payload daily keys:", list(daily.keys()))
            import json
            sample = {k: (daily.get(k)[:3] if isinstance(daily.get(k), list) else daily.get(k)) for k in list(daily.keys())[:10]}
            print(json.dumps(sample, default=str))
        except Exception:
            pass
        raise

# -------------------------
# Main: fetch full-range daily per cell (one request per cell), cache per-cell daily parquet/csv
# -------------------------
n_cells = len(grid_centers)
print(f"Fetching Open‑Meteo daily for {n_cells} cells from {start_date} to {end_date} (one request per cell)")

# pre-create date keys
start_dt = pd.to_datetime(start_date).date()
end_dt = pd.to_datetime(end_date).date()
all_dates = pd.date_range(start=start_dt, end=end_dt, freq="D").date.tolist()
data_by_day = {d.isoformat(): {} for d in all_dates}

def process_cell_daily(idx, lon, lat):
    cell_dir = os.path.join(out_dir, f"cell_{idx:04d}")
    os.makedirs(cell_dir, exist_ok=True)
    base_path = safe_daily_path(cell_dir, idx)
    parquet_path = base_path + ".parquet"
    csv_path = base_path + ".csv"

    # if cached parquet or csv exists, load and return
    if os.path.exists(parquet_path):
        try:
            df_cell = pd.read_parquet(parquet_path)
            if 'date' in df_cell.columns:
                df_cell['date'] = pd.to_datetime(df_cell['date']).dt.date
            return idx, lon, lat, df_cell
        except Exception as e:
            print(f"Warning: failed to read parquet for cell {idx}: {e}")

    if os.path.exists(csv_path):
        try:
            df_cell = pd.read_csv(csv_path, parse_dates=["date"])
            if 'date' in df_cell.columns:
                df_cell['date'] = pd.to_datetime(df_cell['date']).dt.date
            return idx, lon, lat, df_cell
        except Exception as e:
            print(f"Warning: failed to read csv for cell {idx}: {e}")

    # sanity check for coordinate ranges
    if not (-90.0 <= lat <= 90.0 and -180.0 <= lon <= 180.0):
        raise RuntimeError(f"Invalid coordinates for cell {idx}: lat={lat}, lon={lon}")

    # fetch daily range in one request (with robust retry/429 handling)
    payload = fetch_daily_with_retries(lat=lat, lon=lon, start_iso=start_date, end_iso=end_date, daily_vars=daily_vars, timezone=timezone)

    df_cell = parse_daily_payload(payload, daily_vars)

    # if API returned no rows, create full-range empty frame
    if df_cell.shape[0] == 0:
        df_cell = pd.DataFrame({"date": all_dates})
        for var in daily_vars:
            df_cell[var] = np.nan
    else:
        df_cell['date'] = pd.to_datetime(df_cell['date']).dt.date

    # -------------------------
    # Safe reindexing: avoid duplicate 'date' column
    # -------------------------
    if 'date' in df_cell.columns:
        temp_index = pd.to_datetime(df_cell['date'])
        df_cell = df_cell.drop(columns=['date'])
    else:
        temp_index = pd.to_datetime(df_cell.index)

    df_cell = df_cell.set_index(temp_index).reindex(pd.to_datetime(all_dates))
    df_cell.index.name = 'date'
    df_cell = df_cell.reset_index()
    df_cell['date'] = pd.to_datetime(df_cell['date']).dt.date

    # add lon/lat columns
    df_cell['lon'] = lon
    df_cell['lat'] = lat

    # try to save parquet, fallback to CSV
    try:
        df_cell.to_parquet(parquet_path, index=False)
    except Exception as e:
        try:
            df_cell.to_csv(csv_path, index=False)
        except Exception as e2:
            print(f"Failed to save cell {idx} to parquet and csv: {e2}")

    return idx, lon, lat, df_cell

# run cells in parallel (one request per cell)
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(process_cell_daily, idx, lon, lat): (idx, lon, lat) for idx, (lon, lat) in enumerate(grid_centers)}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Fetching cells"):
        idx, lon, lat = futures[fut]
        try:
            res_idx, res_lon, res_lat, df_cell = fut.result()
        except Exception as e:
            print(f"Cell {idx} failed: {e}")
            continue
        if df_cell is None or df_cell.shape[0] == 0:
            continue
        # populate data_by_day from df_cell
        for _, row in df_cell.iterrows():
            date_iso = pd.to_datetime(row['date']).date().isoformat()
            if date_iso not in data_by_day:
                continue
            entry = {'lon': float(res_lon), 'lat': float(res_lat)}
            for var in daily_vars:
                entry[var] = float(row[var]) if var in row and not pd.isna(row[var]) else np.nan
            data_by_day[date_iso][int(res_idx)] = entry

print("Done. Cached per-cell daily files are in:", out_dir)
print("Access daily predictors via data_by_day[date_iso][cell_idx] -> dict of daily features")


Built grid: 21 cols × 25 rows = 525 cells; spacing 2.250 km
Fetching Open‑Meteo daily for 525 cells from 2024-12-01 to 2025-08-31 (one request per cell)


Fetching cells:  95%|█████████▍| 498/525 [00:48<00:27,  1.01s/it]

429 for (35.27664198794009,129.28079263436499) — backing off 2.0s (no Retry-After). Attempt 1/5
429 for (35.29692206287823,129.28085008270318) — backing off 2.0s (no Retry-After). Attempt 1/5
429 for (34.81013242636234,129.3040748423216) — backing off 2.0s (no Retry-After). Attempt 1/5


Fetching cells: 100%|██████████| 525/525 [01:18<00:00,  6.70it/s]

Done. Cached per-cell daily files are in: open_meteo_grid_daily_direct
Access daily predictors via data_by_day[date_iso][cell_idx] -> dict of daily features


In [13]:
data_by_day_far = data_by_day


In [16]:
# ============================================================
# 0. IMPORTS & ASSUMPTIONS
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ASSUMED ALREADY IN MEMORY:
# - grouped: DataFrame with station data
#   columns: ['date','location_id','station_lat','station_lon','value', ... local env predictors ...]
# - data_by_day_far: dict[date_iso][cell_idx] -> {'lon','lat', <daily_vars>}
#   produced by your far-region Open-Meteo script
# - daily_vars: list of Open-Meteo variable names used in data_by_day_far

# ============================================================
# 1. CONFIG
# ============================================================
window_len = 5          # previous 5 days
k_neighbors = 5
sigma = 0.05            # RBF lengthscale (for real-real edges)
ensemble_size = 5

target_col = "value"    # PM2.5(t)
local_feature_cols = [
    "value",
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "wind_speed_10m",
    "wind_direction_10m",
    "surface_pressure",
    "precipitation",
]

start_date = "2024-12-01"
end_date   = "2025-08-31"

# ============================================================
# 2. PREP STATION PANEL (DAILY, SORTED)
# ============================================================
grouped["date"] = pd.to_datetime(grouped["date"])
grouped = grouped.sort_values(["date", "location_id"])

all_dates = sorted(grouped["date"].unique())
all_dates_str = [d.strftime("%Y-%m-%d") for d in all_dates]

usable_dates = [
    d for d in all_dates_str
    if start_date <= d <= end_date and d in data_by_day_far
]

daily_real = {}
for date_val, df_day in grouped.groupby("date"):
    date_str = date_val.strftime("%Y-%m-%d")
    if date_str not in usable_dates:
        continue

    df_day = df_day.sort_values("location_id")
    coords = np.column_stack([df_day["station_lon"].values,
                              df_day["station_lat"].values])  # (N,2)
    y = df_day[target_col].values.astype(np.float32)
    feat = df_day[local_feature_cols].values.astype(np.float32)

    mask = ~np.isnan(y)
    coords = coords[mask]
    y = y[mask]
    feat = feat[mask]

    if coords.shape[0] < k_neighbors + 1:
        continue

    daily_real[date_str] = {
        "coords": coords,
        "feat": feat,
        "y": y
    }

usable_dates = [d for d in usable_dates if d in daily_real]

# ============================================================
# 3. RBF GRAPH HELPERS (REAL-REAL EDGES)
# ============================================================
def rbf_weights(coords, sigma):
    N = coords.shape[0]
    A = np.zeros((N,N), dtype=np.float32)
    for i in range(N):
        d2 = np.sum((coords[i] - coords)**2, axis=1)
        A[i] = np.exp(-d2 / (2*sigma*sigma))
    np.fill_diagonal(A, 0.0)
    return A

def sparsify_knn(A, k):
    N = A.shape[0]
    A_sparse = np.zeros_like(A)
    for i in range(N):
        row = A[i]
        idx = np.argsort(-row)[:k]
        A_sparse[i, idx] = row[idx]
    return 0.5*(A_sparse + A_sparse.T)

# ============================================================
# 4. BUILD BASE SPATIOTEMPORAL TRAIN SAMPLES (REAL NODES ONLY)
# ============================================================
train_samples = []

for idx in range(window_len, len(usable_dates)-1):
    window_dates = usable_dates[idx-window_len:idx]  # t-4..t
    date_t = usable_dates[idx]
    date_tp1 = usable_dates[idx+1]

    g_t = daily_real[date_t]
    coords_t = g_t["coords"]      # (N,2)
    feat_t = g_t["feat"]          # (N,F)
    y_tp1 = daily_real[date_tp1]["y"]  # (N,)

    N = coords_t.shape[0]
    if N < k_neighbors + 1:
        continue

    F_loc = feat_t.shape[1]
    X_st = np.zeros((N, window_len * F_loc), dtype=np.float32)

    for w, d_w in enumerate(window_dates):
        g_w = daily_real[d_w]
        coords_w = g_w["coords"]
        feat_w = g_w["feat"]
        for i in range(N):
            d2 = np.sum((coords_w - coords_t[i])**2, axis=1)
            j = np.argmin(d2)
            X_st[i, w*F_loc:(w+1)*F_loc] = feat_w[j]

    A_full = rbf_weights(coords_t, sigma)
    A = sparsify_knn(A_full, k_neighbors)
    src, dst = np.nonzero(A)

    data = Data(
        x=torch.tensor(X_st, dtype=torch.float32),
        edge_index=torch.tensor(np.vstack([src,dst]), dtype=torch.long),
        y=torch.tensor(y_tp1, dtype=torch.float32),
        coords=torch.tensor(coords_t, dtype=torch.float32),
        date_t=date_t,
        window_dates=window_dates
    )
    train_samples.append(data)

# ============================================================
# 5. FAR-REGION VIRTUAL NODE FEATURES
# ============================================================
far_cells = list(next(iter(data_by_day_far.values())).keys())  # cell indices in far region

def get_far_features_for_window(window_dates, cell_idx):
    feats = []
    for d in window_dates:
        if d not in data_by_day_far:
            feats.append(np.zeros(len(daily_vars), dtype=np.float32))
            continue
        cell = data_by_day_far[d].get(cell_idx)
        if cell is None:
            feats.append(np.zeros(len(daily_vars), dtype=np.float32))
        else:
            feats.append(np.array([cell[var] for var in daily_vars], dtype=np.float32))
    return np.concatenate(feats, axis=0)  # (window_len * len(daily_vars),)

D_real = train_samples[0].x.shape[1]
D_far = window_len * len(daily_vars)

def build_virtual_feature_vec(window_dates, cell_idx):
    v = get_far_features_for_window(window_dates, cell_idx)
    if v.shape[0] < D_real:
        pad = np.zeros(D_real, dtype=np.float32)
        pad[:v.shape[0]] = v
        v = pad
    elif v.shape[0] > D_real:
        v = v[:D_real]
    return v  # (D_real,)

# ============================================================
# 6. BUILD AUGMENTED GRAPH WITH ONE VIRTUAL NODE (FOR GIVEN CELL)
# ============================================================
def build_augmented_graph(data, cell_idx):
    X_real = data.x.cpu().numpy()          # (N,D)
    N, D = X_real.shape

    v_feat = build_virtual_feature_vec(data.window_dates, cell_idx)  # (D,)
    X_aug = np.vstack([X_real, v_feat[None, :]])  # (N+1,D)

    edge_index = data.edge_index.cpu().numpy()

    # connect virtual node (index N) to all real nodes (bidirectional)
    src_v = np.arange(N, dtype=np.int64)
    dst_v = np.full(N, N, dtype=np.int64)

    new_src = np.concatenate([edge_index[0], src_v, dst_v])
    new_dst = np.concatenate([edge_index[1], dst_v, src_v])

    data_aug = Data(
        x=torch.tensor(X_aug, dtype=torch.float32),
        edge_index=torch.tensor(np.vstack([new_src, new_dst]), dtype=torch.long),
        y=data.y.clone()
    )
    return data_aug.to(device)

# ============================================================
# 7. GNN MODEL WITH GATConv
# ============================================================
class GATSpatioTemporal(nn.Module):
    def __init__(self, in_channels, hidden, heads=4):
        super().__init__()
        self.gat1 = GATConv(in_channels, hidden, heads=heads, concat=True)
        self.gat2 = GATConv(hidden*heads, hidden, heads=1, concat=True)
        self.lin = nn.Linear(hidden, 1)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        h = F.elu(self.gat1(x, edge_index))
        h = F.elu(self.gat2(h, edge_index))
        out = self.lin(h).squeeze(-1)
        return out

# ============================================================
# 8. TRAIN DEEP ENSEMBLE WITH VIRTUAL NODE INCLUDED (RANDOM CELL EACH PASS)
# ============================================================
def train_one_model_with_virtual(train_samples, far_cells):
    in_channels = train_samples[0].x.shape[1]
    model = GATSpatioTemporal(in_channels=in_channels, hidden=64).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    for epoch in range(15):
        model.train()
        for data in train_samples:
            # randomly pick a far-region cell for this sample
            cell_idx = np.random.choice(far_cells)
            data_aug = build_augmented_graph(data, cell_idx)

            opt.zero_grad()
            y_hat = model(data_aug)
            N = data.x.shape[0]
            loss = loss_fn(y_hat[:N], data.y.to(device))
            loss.backward()
            opt.step()
    return model

ensemble = [train_one_model_with_virtual(train_samples, far_cells) for _ in range(ensemble_size)]

# ============================================================
# 9. BASELINE MSE (NO VIRTUAL NODE, SAME MODELS)
# ============================================================
def prediction_mse_no_virtual():
    total_mse = 0.0
    count = 0
    for data in train_samples:
        # build graph WITHOUT virtual node
        data_no = Data(
            x=data.x.clone(),
            edge_index=data.edge_index.clone(),
            y=data.y.clone()
        ).to(device)

        preds = []
        for m in ensemble:
            m.eval()
            with torch.no_grad():
                preds.append(m(data_no).cpu().numpy())
        preds = np.stack(preds, axis=0)
        mean_pred = preds.mean(axis=0)
        mse = ((mean_pred - data.y.cpu().numpy())**2).mean()
        total_mse += mse
        count += 1
    return total_mse / max(count,1)

baseline_mse = prediction_mse_no_virtual()
print("Baseline MSE (no virtual node at inference):", baseline_mse)

# ============================================================
# 10. MSE WITH VIRTUAL NODE FOR A GIVEN CELL (GATConv)
# ============================================================
def prediction_mse_with_virtual(cell_idx):
    total_mse = 0.0
    count = 0
    for data in train_samples:
        data_aug = build_augmented_graph(data, cell_idx)
        preds = []
        for m in ensemble:
            m.eval()
            with torch.no_grad():
                preds.append(m(data_aug).cpu().numpy())
        preds = np.stack(preds, axis=0)  # (M, N+1)
        mean_pred = preds.mean(axis=0)
        N = data.x.shape[0]
        mse = ((mean_pred[:N] - data.y.cpu().numpy())**2).mean()
        total_mse += mse
        count += 1
    return total_mse / max(count,1)

# ============================================================
# 11. SEARCH OVER FAR-REGION CELLS FOR BEST VIRTUAL NODE
# ============================================================
print(f"Evaluating {len(far_cells)} far-region cells as virtual node candidates (GATConv)...")

best_cell = None
best_mse = float("inf")

for cell_idx in far_cells:
    mse_with = prediction_mse_with_virtual(cell_idx)
    if mse_with < best_mse:
        best_mse = mse_with
        best_cell = cell_idx

print("Baseline MSE (no virtual node):", baseline_mse)
print("Best MSE with virtual node:", best_mse)
print("Improvement (no - with):", baseline_mse - best_mse)

any_date = next(iter(data_by_day_far.keys()))
best_entry = data_by_day_far[any_date][best_cell]
print("Best virtual node (far-region) cell_idx:", best_cell)
print("Best virtual node lon, lat:", best_entry["lon"], best_entry["lat"])


Baseline MSE (no virtual node at inference): 20387.656
Evaluating 525 far-region cells as virtual node candidates (GATConv)...
Baseline MSE (no virtual node): 20387.656
Best MSE with virtual node: 179.89993
Improvement (no - with): 20207.756
Best virtual node (far-region) cell_idx: 141
Best virtual node lon, lat: 128.93476308617036 35.13484353036436


In [17]:
#additional checks. 
import random
import torch

def run_full_pipeline_once(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # retrain ensemble with virtual node included
    ensemble_stable = [train_one_model_with_virtual(train_samples, far_cells) 
                       for _ in range(ensemble_size)]

    # baseline MSE
    def baseline_mse_fn():
        total, count = 0, 0
        for data in train_samples:
            data_no = Data(
                x=data.x.clone(),
                edge_index=data.edge_index.clone(),
                y=data.y.clone()
            ).to(device)

            preds = []
            for m in ensemble_stable:
                m.eval()
                with torch.no_grad():
                    preds.append(m(data_no).cpu().numpy())
            mean_pred = np.mean(preds, axis=0)
            total += ((mean_pred - data.y.cpu().numpy())**2).mean()
            count += 1
        return total / count

    baseline = baseline_mse_fn()

    # search best virtual node
    best_cell, best_mse = None, float("inf")
    for cell_idx in far_cells:
        mse = prediction_mse_with_virtual(cell_idx)
        if mse < best_mse:
            best_mse = mse
            best_cell = cell_idx

    return baseline, best_mse, best_cell

# ---- run stability test ----
stability_results = []
for seed in [0, 1, 2, 3, 4]:
    baseline, best_mse, best_cell = run_full_pipeline_once(seed)
    stability_results.append((seed, baseline, best_mse, baseline - best_mse, best_cell))

stability_results


[(0, np.float32(208423.44), np.float32(179.89993), np.float32(208243.53), 141),
 (1, np.float32(21038.516), np.float32(179.89993), np.float32(20858.615), 141),
 (2, np.float32(10124.183), np.float32(179.89993), np.float32(9944.282), 141),
 (3, np.float32(872913.7), np.float32(179.89993), np.float32(872733.8), 141),
 (4, np.float32(22818.125), np.float32(179.89993), np.float32(22638.225), 141)]

In [ ]:
# ---- split dates ----
num_dates = len(train_samples)
cut = int(0.8 * num_dates)

train_split = train_samples[:cut]
test_split  = train_samples[cut:]

# ---- retrain ensemble on TRAIN ONLY ----
ensemble_holdout = [train_one_model_with_virtual(train_split, far_cells)
                    for _ in range(ensemble_size)]

# ---- baseline on TEST ----
def baseline_mse_test():
    total, count = 0, 0
    for data in test_split:
        data_no = Data(
            x=data.x.clone(),
            edge_index=data.edge_index.clone(),
            y=data.y.clone()
        ).to(device)

        preds = []
        for m in ensemble_holdout:
            m.eval()
            with torch.no_grad():
                preds.append(m(data_no).cpu().numpy())
        mean_pred = np.mean(preds, axis=0)
        total += ((mean_pred - data.y.cpu().numpy())**2).mean()
        count += 1
    return total / count

baseline_test = baseline_mse_test()

# ---- evaluate each virtual node on TEST ----
best_cell_test, best_mse_test = None, float("inf")
for cell_idx in far_cells:
    total, count = 0, 0
    for data in test_split:
        data_aug = build_augmented_graph(data, cell_idx)
        preds = []
        for m in ensemble_holdout:
            m.eval()
            with torch.no_grad():
                preds.append(m(data_aug).cpu().numpy())
        mean_pred = np.mean(preds, axis=0)
        N = data.x.shape[0]
        total += ((mean_pred[:N] - data.y.cpu().numpy())**2).mean()
        count += 1
    mse = total / count
    if mse < best_mse_test:
        best_mse_test = mse
        best_cell_test = cell_idx

baseline_test, best_mse_test, baseline_test - best_mse_test, best_cell_test


In [ ]:
#train the GAT without the additional node. 
def train_one_model_no_virtual(train_samples):
    in_channels = train_samples[0].x.shape[1]
    model = GATSpatioTemporal(in_channels=in_channels, hidden=64).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    for epoch in range(15):
        model.train()
        for data in train_samples:
            data_no = Data(
                x=data.x.clone(),
                edge_index=data.edge_index.clone(),
                y=data.y.clone()
            ).to(device)

            opt.zero_grad()
            y_hat = model(data_no)
            loss = loss_fn(y_hat, data.y.to(device))
            loss.backward()
            opt.step()
    return model

ensemble_ablation = [train_one_model_no_virtual(train_samples)
                     for _ in range(ensemble_size)]
